# Notebook 09: 强化学习交易智能体 -- 山东 15min 现货市场
## RL Trading Agent on Shandong 15-Minute Spot Market

**目标**: 理解 Gymnasium RL 环境接口，用 ElectricityMarketEnv 构建 96 维交易环境，
训练 PPO 智能体自动投标，掌握训练曲线和动作分布分析方法。

**数据源**: 山东电力现货 15 分钟数据 (2024-01~2026-01)，96 点/日。

> PPO (Proximal Policy Optimization) 是 OpenAI 2017 年提出的策略梯度算法，
> 通过 clipped surrogate objective 限制策略更新步长，是目前最稳定的 on-policy RL 算法。

In [ ]:
# -- 导入依赖 / Imports --
import os, sys, warnings, logging
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

import gymnasium as gym
from stable_baselines3.common.monitor import Monitor

from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data
from ellectric.pipeline.features import FeatureEngineer
from ellectric.pipeline.forecaster import XGBoostForecaster
from ellectric.pipeline.trading_env import ElectricityMarketEnv, RewardRegistry
from ellectric.pipeline.rl_trainer import RLAgentFactory
from ellectric.config import TimeConfig

print("All imports OK / 所有依赖导入成功")
print(f"  TimeConfig.points_per_day = {TimeConfig.points_per_day} (15min -> 96 点/日)")
print(f"  RewardRegistry: {RewardRegistry.list()}")

---
## 步骤 1: 加载山东 15min 数据 / Step 1: Load Shandong Data

用 2 个月数据 (2024-07~2024-08) 避免训练过长。
`ElectricityMarketEnv` 要求 `price_data` 有 `price_da` 列，山东数据提供 `da_price`，重命名即可。

In [ ]:
# -- 加载数据 / Load --
loader = create_loader("shandong")
df = loader.load_data("2024-07-01", "2024-08-31")
df = clean_data(df)

# 构建负荷和价格 DataFrame (env 要求 price_da 列)
df_load = df[["timestamp", "load_mw"]].copy()
df_price = df[["timestamp", "load_mw", "da_price", "rt_price",
               "wind_actual_mw", "solar_actual_mw"]].copy()
df_price = df_price.rename(columns={"da_price": "price_da"})

print(f"Loaded / 已加载: {len(df)} rows")
print(f"  Time range / 时间: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"  load_mw: {df['load_mw'].mean():.0f} +/- {df['load_mw'].std():.0f} MW")
print(f"  price_da: {df_price['price_da'].mean():.1f} +/- {df_price['price_da'].std():.1f} 元/MWh")

---
## 步骤 2: 特征工程 + 训练 XGBoost 负荷预测器 / Step 2: Feature Engineering + XGBoost

ElectricityMarketEnv 需要负荷预测器生成观测。训练快速 XGBoost (~30 秒)。

In [ ]:
# -- 特征工程 / Feature Engineering (Tier 1) --
fe = FeatureEngineer()
df_feat = fe.add_tier1_features(df_load)
feature_cols = [c for c in df_feat.columns if c not in ("timestamp", "load_mw")]
print(f"Features / 特征 ({len(feature_cols)}): {feature_cols}")

# 跳过 NaN (lag_96 引起的)
lag_skip = TimeConfig.points_per_day
X = df_feat[feature_cols].iloc[lag_skip:]
y = df_feat["load_mw"].iloc[lag_skip:]

# -- 训练 XGBoost --
lf = XGBoostForecaster(n_estimators=50, max_depth=4, learning_rate=0.1)
xgb_result = lf.train_evaluate(X, y, n_splits=3, gap=TimeConfig.points_per_day)
m = xgb_result["metrics"]
print(f"XGBoost / 训练完成: MAE={m['mae']:.1f} MW, RMSE={m['rmse']:.1f} MW, R2={m.get('r2',0):.3f}")

---
## 步骤 3: 初始化 96 维 ElectricityMarketEnv / Step 3: 96-Dim Trading Environment

观测空间 (Dict): `load_forecast`(96,), `price_forecast`(96,), `time_features`(4,),
`price_history`(672,), `account_state`(2,)。
动作空间: `Box(0, 1, (96,))` -- 每 15 分钟归一化投标量。

In [ ]:
# -- 初始化环境 + 随机策略基线 / Init env + Random baseline --
env = ElectricityMarketEnv(
    load_data=df_load, price_data=df_price,
    load_forecaster=lf, price_forecaster=None,
    initial_cash=0.0, max_capacity=df_load["load_mw"].max(),
    reward_fn="profit_only",
)
print(f"Action dim / 动作维度: {env.action_space.shape[0]} (96 = 15min/day)")
print(f"Max capacity / 最大容量: {env._max_capacity:.0f} MW")

# -- 验证 reset/step --
obs, info = env.reset()
for k, v in obs.items():
    print(f"  obs['{k}']: shape={v.shape}")
action = env.action_space.sample()
obs, reward, _, _, info = env.step(action)
print(f"  Step reward: {reward:.4f}, info keys: {list(info.keys())}")

# -- 随机策略 100 episodes 基线 --
N_RANDOM = 100; random_rewards = []
for ep in range(N_RANDOM):
    obs, _ = env.reset(); done = False; total = 0.0
    while not done:
        obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
        total += r; done = terminated or truncated
    random_rewards.append(total)
random_rewards = np.array(random_rewards)
print(f"\nRandom baseline / 随机策略基线: mean={random_rewards.mean():.2f}, std={random_rewards.std():.2f}")

# -- 随机策略奖励分布 / Reward Distribution --
fig = go.Figure()
fig.add_trace(go.Histogram(x=random_rewards, nbinsx=25,
    marker=dict(color="#1f77b4", line=dict(color="white", width=1))))
fig.add_vline(x=random_rewards.mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mean {random_rewards.mean():.2f}")
fig.update_layout(title="Random Policy Reward Distribution / 随机策略奖励分布 (100 eps)",
    xaxis_title="Episode Reward", yaxis_title="Count", height=380, showlegend=False)
fig.show()

---
## 步骤 4: PPO 训练 / Step 4: PPO Training

训练 96 维 PPO 智能体。可调整 `TOTAL_TIMESTEPS` 控制训练时间。
> 预计: CPU 5-15 分钟。可减至 10000 快速测试。

In [ ]:
# -- PPO 训练 / PPO Training --
TOTAL_TIMESTEPS = 30_000
TB_LOG = "../tb_logs"; os.makedirs(TB_LOG, exist_ok=True)
train_env = Monitor(env, filename=f"{TB_LOG}/monitor.csv")

agent = RLAgentFactory.create("ppo", train_env, tensorboard_log=TB_LOG,
    verbose=1, learning_rate=0.0003, n_steps=2048, batch_size=64, n_epochs=10)
print(f"Starting PPO / 开始训练 ({TOTAL_TIMESTEPS} timesteps)...")
result = agent.train(total_timesteps=TOTAL_TIMESTEPS)

print(f"\nPPO done / 训练完成: final_reward={result['final_reward']:.2f}")
os.makedirs("../models", exist_ok=True)
agent.save("../models/ppo_shandong_96d.zip")
print(f"Saved / 已保存: ../models/ppo_shandong_96d.zip")

In [ ]:
# -- 训练曲线 + 评估 / Training Curves + Evaluation --
ep_rew = train_env.get_episode_rewards()
print(f"Episodes completed / 完成回合: {len(ep_rew)}")

if len(ep_rew) > 0:
    window = max(1, len(ep_rew) // 15)
    smoothed = pd.Series(ep_rew).rolling(window=window, min_periods=1).mean()

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
        subplot_titles=("Episode Reward / 每回合奖励", "Episode Length / 回合长度"),
        row_heights=[0.6, 0.4])
    fig.add_trace(go.Scatter(y=ep_rew, mode="markers", name="Reward",
        marker=dict(color="#1f77b4", size=3, opacity=0.5)), row=1, col=1)
    fig.add_trace(go.Scatter(y=smoothed, mode="lines", name=f"SMA({window})",
        line=dict(color="#d62728", width=2)), row=1, col=1)
    mean_rnd = float(random_rewards.mean())
    fig.add_hline(y=mean_rnd, line_dash="dot", line_color="gray",
                  annotation_text=f"Random mean {mean_rnd:.1f}", row=1, col=1)
    fig.add_trace(go.Scatter(y=train_env.get_episode_lengths(), mode="markers",
        marker=dict(color="#2ca02c", size=3, opacity=0.5)), row=2, col=1)
    fig.update_layout(height=500, hovermode="x unified")
    fig.update_xaxes(title_text="Episode", row=2, col=1)
    fig.update_yaxes(title_text="Reward", row=1, col=1)
    fig.update_yaxes(title_text="Steps", row=2, col=1)
    fig.show()

    last10 = np.mean(ep_rew[-10:]) if len(ep_rew) >= 10 else np.mean(ep_rew)
    print(f"Last 10 avg / 最后10回合: {last10:.2f} | Random / 随机: {mean_rnd:.2f} | Gain / 提升: {last10 - mean_rnd:+.2f}")

---
## 步骤 5: 评估 + 动作分布分析 / Step 5: Evaluation + Action Distribution

运行 5 个 episode 查看实际负荷 vs 投标量。收集 10 个 episode 的动作分析投标偏好。

In [ ]:
# -- 评估 5 个 episode / Evaluate 5 episodes --
eval_env = ElectricityMarketEnv(
    load_data=df_load, price_data=df_price, load_forecaster=lf, price_forecaster=None,
    initial_cash=0.0, max_capacity=df_load["load_mw"].max(), reward_fn="profit_only")

eval_results = []
for ep in range(5):
    obs, _ = eval_env.reset(); done = False
    ep_data = {"actuals": [], "bids": [], "rewards": [], "prices": []}
    while not done:
        action = agent.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_data["actuals"].extend(info.get("actual_load", []))
        ep_data["bids"].extend(info.get("bid_mw", []))
        ep_data["rewards"].append(reward)
        ep_data["prices"].extend(info.get("clearing_price", []))
        done = terminated or truncated
    eval_results.append(ep_data)
    print(f"  Ep {ep+1}: total_reward={np.sum(ep_data['rewards']):.2f}")

# -- 可视化第一个 episode / Plot first episode (first 2 days) --
ep0 = eval_results[0]; n = min(192, len(ep0["actuals"]))
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("Load vs Bid / 负荷 vs 投标", "Per-Step Reward", "Clearing Price / 出清价格"),
    row_heights=[0.45, 0.25, 0.3])
x = list(range(n))
fig.add_trace(go.Scatter(x=x, y=ep0["actuals"][:n], mode="lines",
    name="Actual / 实际", line=dict(color="#1f77b4", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=ep0["bids"][:n], mode="lines",
    name="PPO Bid / 投标", line=dict(color="#ff7f0e", width=2, dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(y=ep0["rewards"], mode="lines+markers", name="Reward",
    marker=dict(size=4, color="#2ca02c")), row=2, col=1)
fig.add_trace(go.Scatter(x=x, y=ep0["prices"][:n], mode="lines", name="Price",
    line=dict(color="#9467bd", width=1.5)), row=3, col=1)
fig.update_layout(height=600, hovermode="x unified")
fig.update_xaxes(title_text="15-min Step", row=3, col=1)
fig.update_yaxes(title_text="MW", row=1, col=1)
fig.update_yaxes(title_text="Reward", row=2, col=1)
fig.update_yaxes(title_text="Price / 元/MWh", row=3, col=1)
fig.show()

In [ ]:
# -- 动作分布分析（按小时聚合）/ Action Distribution (aggregated by hour) --
all_actions = []
eval_env2 = ElectricityMarketEnv(
    load_data=df_load, price_data=df_price, load_forecaster=lf, price_forecaster=None,
    initial_cash=0.0, max_capacity=df_load["load_mw"].max(), reward_fn="profit_only")

for _ in range(10):
    obs, _ = eval_env2.reset(); done = False
    while not done:
        all_actions.append(agent.predict(obs, deterministic=True).tolist())
        obs, _, terminated, truncated, _ = eval_env2.step(agent.predict(obs, deterministic=True))
        done = terminated or truncated
all_actions = np.array(all_actions)

# 按小时聚合（4*15min = 1h）
n_periods = all_actions.shape[1]
hourly_means = [np.mean(all_actions[:, i:i+4]) for i in range(0, n_periods, 4)]
hourly_stds = [np.std(all_actions[:, i:i+4]) for i in range(0, n_periods, 4)]
hour_labels = [f"{h:02d}:00" for h in range(24)]

fig = go.Figure()
fig.add_trace(go.Bar(x=hour_labels, y=hourly_means,
    error_y=dict(type="data", array=hourly_stds, visible=True),
    marker=dict(color=hourly_means, colorscale="Viridis", showscale=True,
                colorbar=dict(title="Avg Bid"))))
fig.update_layout(title="PPO Action Distribution by Hour / 每小时平均投标量",
    xaxis=dict(title="Time of Day", tickangle=-45),
    yaxis=dict(title="Normalized Bid [0,1]", range=[0, 1]), height=400, showlegend=False)
fig.show()

print("Hourly mean bids / 各小时平均投标:")
for h in range(24):
    print(f"  {hour_labels[h]}: {hourly_means[h]:.3f} +/- {hourly_stds[h]:.3f}")

---
## 分析与思考 / Analysis & Discussion

### 投标模式 / Bid Patterns
- **高峰时段 (8-20h)**: 负荷高，投标量通常较高
- **凌晨时段 (0-6h)**: 负荷低，投标量应减少
- **标准差**: 反映策略对不同场景的自适应能力

> 如果 PPO 在所有 96 点都输出相近值，说明策略没学到时间模式 -- 需增加训练步数。

### 思考题 / Questions
1. PPO reward 比随机策略高多少？96 维 vs 24 维对收敛有何影响？
2. 训练曲线有稳定上升趋势吗？如果没有，可能原因是什么？
3. 动作分布图显示 PPO 是否学会「低谷少投、高峰多投」？
4. 若换成 `risk_adjusted` 或 `volume_penalty` reward，策略行为会如何变化？

---
**下一步 / Next**: Notebook 10 -- 多算法对比 (PPO vs TD3 vs SAC) + 回测
Notebook 11 -- SHAP 模型可解释性